In [2]:
import torch

In [25]:
# x = 2.0
x1 = torch.Tensor([2.0]).double() ; x1.requires_grad = True

# w = -3.0
w1 = torch.Tensor([-3.0]).double() ; w1.requires_grad = True

# b = 5
b = torch.Tensor([5]).double() ; b.requires_grad = True

n = x1 * w1 + b
print(n.data)
print('n', n.data.item())

n.backward()
print('---')

# partial derivatives of these variables wrt n
print('x1', x1.grad.item())
print('w1', w1.grad.item())

tensor([-1.], dtype=torch.float64)
n -1.0
---
x1 -3.0
w1 2.0


In [5]:
import math
import numpy as np

In [41]:
# grad = 0.0 until backward is called

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda : None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f'data={self.data}, grad={self.grad}'
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad * 1.0
            other.grad += out.grad * 1.0

        out._backward = _backward
        return out
    
    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data - other.data, (self, other), '-')

        def _backward():
            self.grad += out.grad * 1.0
            other.grad += out.grad * -1.0

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def topo_sort(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    topo_sort(child)
                topo.append(node)

        topo_sort(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
    
v1 = Value(5.0)
v2 = Value(3.0)
v3 = v1 + v2
v3.backward()
print(v1)
print(v2)


data=5.0, grad=1.0
data=3.0, grad=1.0
